# LightGBM GBDT Pairwise Ranker - Smart MCQ Solver (With Wandb Logging)

### Overview and Wandb Tracking Features
This notebook implements the **LightGBM Gradient Boosting Decision Tree (GBDT) Pairwise Ranker** with **Fold-Wise and Metric-Wise Weights and Biases (Wandb) Logging**.

- **Primary Kaggle Metric**: MAP@3 (Mean Average Precision at 3)
- **Secondary Evaluation Metrics**:
  - Top-1 Accuracy: Standard classification accuracy (single best prediction).
  - Top-3 Accuracy: Recall metric checking if ground truth is within top 3 ranked answers.
  - F1-Score (Macro): Harmonic mean of precision and recall.
  - Precision (Macro): Proportion of correct positive predictions.
  - Recall (Macro): Coverage of actual ground truth labels.
  - Log Loss: Binary Cross Entropy of option probability predictions.

- **Wandb Dashboard Logs**:
  - Hyperparameter Configuration: Track n_estimators, learning_rate, num_leaves, colsample_bytree, max_features, n_splits.
  - Per-Fold Metric Progression: Logs fold/val_loss, fold/val_map3, fold/val_top1_accuracy, fold/val_top3_accuracy, fold/val_f1_score, fold/val_precision, fold/val_recall.
  - Comparative Model Summary Table: Automatically generates a wandb.Table comparing LightGBM performance metrics across all folds.

---


In [1]:
# Cell 1: Core Imports and Setup
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

warnings.filterwarnings('ignore')
print("Core libraries imported successfully!")


Core libraries imported successfully!


In [2]:
# Cell 2: Weights and Biases (Wandb) Setup
try:
    import wandb
    WANDB_API_KEY = "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"
    WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
    WANDB_PROJECT = "23f3001092-t22026"

    wandb.login(key=WANDB_API_KEY)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb running in offline fallback mode ({e})")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Wandb initialized successfully!


In [3]:
# Cell 3: Dataset Loading and Setup
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path = next((p for p in test_candidates if os.path.exists(p)), test_candidates[0])
    
    df_train = pd.read_csv(train_path).fillna('')
    df_test = pd.read_csv(test_path).fillna('')
    
    return df_train, df_test

train, test = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
train.head(3)


Train shape: (2000, 8)
Test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [4]:
# Cell 4: Evaluation Metric Functions
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def compute_all_metrics(actuals, pred_lists):
    map3_score = mapk(actuals, pred_lists)
    top1_preds = [p[0] for p in pred_lists]
    top1_acc = accuracy_score(actuals, top1_preds)
    top3_acc = np.mean([1 if a in p[:3] else 0 for a, p in zip(actuals, pred_lists)])
    prec, rec, f1, _ = precision_recall_fscore_support(actuals, top1_preds, average='macro', zero_division=0)
    
    return {
        'map3': map3_score,
        'top1_accuracy': top1_acc,
        'top3_accuracy': top3_acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }

def rank_preds(pw, scores, orig_df):
    pw_temp = pw.copy()
    pw_temp['score'] = scores
    
    ranked = pw_temp.sort_values(['qid', 'score'], ascending=[True, False])
    grouped = ranked.groupby('qid')['option'].apply(lambda opts: list(opts)[:3]).to_dict()
    
    return [grouped.get(qid, ['A', 'B', 'C']) for qid in orig_df['id']]


In [5]:
# Cell 5: Pairwise Data Preparation and Lexical Feature Extraction
def make_pairwise_dataset(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        p_words = set(re.findall(r'\w+', prompt.lower()))
        ans = r.get('answer', None)
        
        for c in OPTION_COLS:
            opt_str = str(r[c])
            o_words = set(re.findall(r'\w+', opt_str.lower()))
            
            overlap = len(p_words.intersection(o_words))
            jaccard = overlap / max(len(p_words.union(o_words)), 1)
            len_ratio = len(opt_str) / max(len(prompt), 1)
            
            combined_text = f"{prompt} [SEP] {opt_str}"
            
            row = {
                'qid': qid,
                'option': c,
                'text': combined_text,
                'overlap': overlap,
                'jaccard': jaccard,
                'len_ratio': len_ratio
            }
            if is_train:
                row['label'] = 1 if (ans == c) else 0
            rows.append(row)
            
    return pd.DataFrame(rows)

pw_train = make_pairwise_dataset(train, is_train=True)
pw_test = make_pairwise_dataset(test, is_train=False)

print(f"Pairwise Train Shape: {pw_train.shape}")
print(f"Pairwise Test Shape : {pw_test.shape}")
pw_train.head(3)


Pairwise Train Shape: (10000, 7)
Pairwise Test Shape : (2500, 6)


,qid,option,text,overlap,jaccard,len_ratio,label
0,1,A,Pick the best possible answer: What is Martin ...,8,0.153846,2.161972,0
1,1,B,Pick the best possible answer: What is Martin ...,7,0.148936,1.816901,1
2,1,C,Pick the best possible answer: What is Martin ...,10,0.250000,1.563380,0


In [6]:
# Cell 6: Word and Character TF-IDF Matrix Construction
print("Fitting Word and Character TF-IDF Vectorizers...")

tfidf_word = TfidfVectorizer(max_features=40000, ngram_range=(1, 2), sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(max_features=30000, ngram_range=(2, 4), sublinear_tf=True, analyzer='char_wb')

combined_corpus = pd.concat([pw_train['text'], pw_test['text']])
tfidf_word.fit(combined_corpus)
tfidf_char.fit(combined_corpus)

Xw_tr = tfidf_word.transform(pw_train['text'])
Xw_te = tfidf_word.transform(pw_test['text'])

Xc_tr = tfidf_char.transform(pw_train['text'])
Xc_te = tfidf_char.transform(pw_test['text'])

X_num_tr = csr_matrix(pw_train[['overlap', 'jaccard', 'len_ratio']].values)
X_num_te = csr_matrix(pw_test[['overlap', 'jaccard', 'len_ratio']].values)

X_tr = hstack([Xw_tr, Xc_tr, X_num_tr]).tocsr().astype(np.float32)
X_te = hstack([Xw_te, Xc_te, X_num_te]).tocsr().astype(np.float32)

y_tr = pw_train['label'].values
groups = pw_train['qid'].values

print(f"Train Feature Matrix: {X_tr.shape}")
print(f"Test Feature Matrix : {X_te.shape}")


Fitting Word and Character TF-IDF Vectorizers...
Train Feature Matrix: (10000, 26959)
Test Feature Matrix : (2500, 26959)


In [7]:
# Cell 7: 5-Fold LightGBM Training Loop with Fold-Wise Wandb Logging
gkf = GroupKFold(n_splits=5)

oof_lgbm = np.zeros(len(pw_train))
tst_lgbm = np.zeros(len(pw_test))

config_dict = {
    "architecture": "LightGBM GBDT Pairwise Ranker",
    "dataset": "Smart MCQ Solver Challenge",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "colsample_bytree": 0.8,
    "subsample": 0.8,
    "max_features_word": 40000,
    "max_features_char": 30000,
    "n_splits": 5
}

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="scrach-lgbm-detailed-fold-logs",
        config=config_dict
    )

print("\nStarting 5-Fold LightGBM Cross Validation...")

fold_metrics_list = []

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        colsample_bytree=0.8,
        subsample=0.8,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
    
    model.fit(
        X_tr[ti], y_tr[ti],
        eval_set=[(X_tr[vi], y_tr[vi])],
        eval_metric='binary_logloss'
    )
    
    val_probs = model.predict_proba(X_tr[vi])[:, 1]
    oof_lgbm[vi] = val_probs
    tst_lgbm += model.predict_proba(X_te)[:, 1] / 5.0
    
    val_qids = pw_train.iloc[vi]['qid'].unique()
    val_df_orig = train[train['id'].isin(val_qids)].reset_index(drop=True)
    val_pw_df = pw_train[pw_train['qid'].isin(val_qids)].reset_index(drop=True)
    
    fold_preds = rank_preds(val_pw_df, val_probs, val_df_orig)
    fold_actuals = val_df_orig['answer'].tolist()
    
    m = compute_all_metrics(fold_actuals, fold_preds)
    clipped_probs = np.clip(val_probs, 1e-15, 1 - 1e-15)
    fold_val_loss = log_loss(val_pw_df['label'], clipped_probs)
    
    print(f" Fold {fold+1} Loss: {fold_val_loss:.4f} | MAP@3: {m['map3']:.4f} | Acc: {m['top1_accuracy']:.4f} | F1: {m['f1_score']:.4f}")
    
    fold_metrics_list.append(m)
    
    if USE_WANDB:
        wandb.log({
            "fold": fold + 1,
            "fold/val_loss": fold_val_loss,
            "fold/val_map3": m['map3'],
            "fold/val_top1_accuracy": m['top1_accuracy'],
            "fold/val_top3_accuracy": m['top3_accuracy'],
            "fold/val_f1_score": m['f1_score'],
            "fold/val_precision": m['precision'],
            "fold/val_recall": m['recall']
        })

overall_actuals = train['answer'].tolist()
m_lgbm = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_lgbm, train))
clipped_oof_probs = np.clip(oof_lgbm, 1e-15, 1 - 1e-15)
oof_loss = log_loss(pw_train['label'], clipped_oof_probs)

print("\n==================================================")
print("       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY")
print("==================================================")
print(f" Primary Kaggle Metric (MAP@3): {m_lgbm['map3']:.4f}")
print(f" Top-1 Accuracy               : {m_lgbm['top1_accuracy']:.4f}")
print(f" Top-3 Recall Accuracy        : {m_lgbm['top3_accuracy']:.4f}")
print(f" Macro F1 Score               : {m_lgbm['f1_score']:.4f}")
print(f" Macro Precision              : {m_lgbm['precision']:.4f}")
print(f" Macro Recall                 : {m_lgbm['recall']:.4f}")
print(f" Pairwise Log Loss            : {oof_loss:.4f}")
print("==================================================\n")

if USE_WANDB:
    wandb.log({
        "oof/overall_map3": m_lgbm['map3'],
        "oof/overall_top1_accuracy": m_lgbm['top1_accuracy'],
        "oof/overall_top3_accuracy": m_lgbm['top3_accuracy'],
        "oof/overall_f1_score": m_lgbm['f1_score'],
        "oof/overall_precision": m_lgbm['precision'],
        "oof/overall_recall": m_lgbm['recall'],
        "oof/overall_val_loss": oof_loss
    })
    
    table = wandb.Table(columns=["Model Architecture", "MAP@3 Score", "Top-1 Accuracy", "Top-3 Accuracy", "F1 Score", "Precision", "Recall", "Log Loss"])
    table.add_data("LightGBM GBDT Ranker", m_lgbm['map3'], m_lgbm['top1_accuracy'], m_lgbm['top3_accuracy'], m_lgbm['f1_score'], m_lgbm['precision'], m_lgbm['recall'], oof_loss)
    
    wandb.log({"model_comparative_summary": table})
    wandb.finish()


wandb: setting up run jt3igujt
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260727_175146-jt3igujt
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scrach-lgbm-detailed-fold-logs
wandb: ⭐️ View project at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: 🚀 View run at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/jt3igujt



Starting 5-Fold LightGBM Cross Validation...

--- Fold 1/5 ---
 Fold 1 Loss: 0.0485 | MAP@3: 0.9962 | Acc: 0.9925 | F1: 0.9918

--- Fold 2/5 ---
 Fold 2 Loss: 0.0517 | MAP@3: 0.9925 | Acc: 0.9850 | F1: 0.9859

--- Fold 3/5 ---
 Fold 3 Loss: 0.0540 | MAP@3: 0.9962 | Acc: 0.9925 | F1: 0.9920

--- Fold 4/5 ---
 Fold 4 Loss: 0.0467 | MAP@3: 0.9950 | Acc: 0.9900 | F1: 0.9901

--- Fold 5/5 ---
 Fold 5 Loss: 0.0531 | MAP@3: 0.9950 | Acc: 0.9900 | F1: 0.9904

       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY
 Primary Kaggle Metric (MAP@3): 0.9950
 Top-1 Accuracy               : 0.9900
 Top-3 Recall Accuracy        : 1.0000
 Macro F1 Score               : 0.9900
 Macro Precision              : 0.9915
 Macro Recall                 : 0.9888
 Pairwise Log Loss            : 0.0508



wandb: uploading artifact run-jt3igujt-model_comparative_summary; updating run metadata
wandb: uploading artifact run-jt3igujt-model_comparative_summary
wandb: uploading media/table/model_comparative_summary_6_a6d3723c83a53fe2976e.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 4-6, summary, console lines 16-28
wandb: 
wandb: Run history:
wandb:                   fold ▁▃▅▆█
wandb:      fold/val_f1_score █▁█▆▆
wandb:          fold/val_loss ▃▆█▁▇
wandb:          fold/val_map3 █▁█▆▆
wandb:     fold/val_precision █▁▇▇▅
wandb:        fold/val_recall ▇▁█▅▇
wandb: fold/val_top1_accuracy █▁█▆▆
wandb: fold/val_top3_accuracy ▁▁▁▁▁
wandb:   oof/overall_f1_score ▁
wandb:       oof/overall_map3 ▁
wandb:                     +5 ...
wandb: 
wandb: Run summary:
wandb:                   fold 5
wandb:      fold/val_f1_score 0.99036
wandb:          fold/val_loss 0.05314
wandb:          fold/val_map3 0.995
wandb:     fold/val_precision 0.

In [8]:
# Cell 8: Final Submission Generation and Comparative Report Table
test_preds = rank_preds(pw_test, tst_lgbm, test)

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

report_df = pd.DataFrame([
    {"Model Architecture": "LightGBM GBDT Pairwise Ranker", "MAP@3": f"{m_lgbm['map3']:.4f}", "Top-1 Acc": f"{m_lgbm['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_lgbm['top3_accuracy']:.4f}", "F1 Score": f"{m_lgbm['f1_score']:.4f}", "Log Loss": f"{oof_loss:.4f}"}
])

print("\n" + "="*70)
print("          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION")
print("="*70)
print(report_df.to_string(index=False))
print("="*70 + "\n")

print("Submission Sample:")
submission.head(10)


Successfully generated submission.csv!

          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION
           Model Architecture  MAP@3 Top-1 Acc Top-3 Acc F1 Score Log Loss
LightGBM GBDT Pairwise Ranker 0.9950    0.9900    1.0000   0.9900   0.0508

Submission Sample:


,ID,Prediction
0,1,A D C
1,2,B A E
2,3,B D E
3,4,E C D
4,5,C D B
5,6,D A C
6,7,E D C
7,8,B E C
8,9,C D E
9,10,B D A
